# Signaux finance et marchés pour anticiper les crises de supply chain

**Projet 6 – ESME Sudria · agent finance.** Ce notebook extrait des données financières et réglementaires gratuites, fixe un seuil d'alerte propre à chaque signal, puis vérifie sur des crises de fournisseurs réelles **si le signal bougeait avant que le client soit touché, et avec combien de jours d'avance**.

**La question** : quels signaux annoncent qu'un fournisseur va coûter plus cher, livrer en retard ou ne plus livrer du tout ?

**Démarche**
1. Extraire les cinq familles de signaux : marchés, fournisseurs cotés, santé des fournisseurs, règles et taxes, sanctions et propriété.
2. Contrôler la qualité des données.
3. Fixer un seuil propre à chaque signal (calibré sur 2010-2020).
4. Valider sur quatre crises de fournisseurs : Nyrstar (2022), fournisseurs de Tesla Grünheide (2024), Wolfspeed (2025), Nexperia (2025).
5. Compter le bruit (alertes par an).
6. Mesurer le délai de transmission Brent → gazole.
7. Mode live (marchés + fiche de chaque fournisseur) et exports pour le moteur de risque.

Toute la collecte passe par `finance_collecte.py`, avec un cache disque : une même requête n'est envoyée qu'une fois.

## 0. Les sources de données utilisées

| Famille | Source | Ce qu'elle donne | Fréquence | Accès |
| --- | --- | --- | --- | --- |
| Marchés | **yfinance** (contrats à terme) | Brent, cuivre, aluminium | Quotidienne | Automatique |
| Marchés | **Banque mondiale** (Pink Sheet) | Gaz naturel Europe, zinc, nickel | Mensuelle | Automatique |
| Marchés | **Weekly Oil Bulletin** (Commission européenne) | Gazole toutes taxes par pays de l'UE | Hebdomadaire | Automatique |
| Marchés | **GSCPI** (Réserve fédérale de New York) | Pression mondiale sur les chaînes d'approvisionnement | Mensuelle | Automatique |
| Marchés | **BCE** | Taux EUR/USD et EUR/CNY | Quotidienne | Automatique |
| Marchés | **Eurostat** (`sts_inpp_m`) | Prix de production de l'industrie dans l'UE | Mensuelle | Automatique |
| Marchés | **ICAP** (carbone EU ETS), **Ember** (électricité) | Prix du carbone, prix de gros de l'électricité | Quotidienne | Fichier CSV placé dans `data/manuel/` |
| Fournisseurs cotés | **yfinance** (actions) | Cours de bourse de chaque fournisseur coté | Quotidienne | Automatique |
| Santé des fournisseurs | **BODACC** | Procédures collectives des entreprises françaises | Quotidienne | Automatique |
| Règles et taxes | **Federal Register** (États-Unis) | Entity List, droits de douane | Quotidienne | Automatique |
| Règles et taxes | **Règles UE** (EUR-Lex, CBAM, droits antidumping) | Nouvelles taxes et interdictions dans l'UE | Au fil de l'eau | Tableau tenu à la main, sourcé |
| Sanctions et propriété | **Sanctions financières de l'UE**, **Consolidated Screening List** (US) | Entreprises sanctionnées, avec date d'inscription | Quotidienne | Automatique |
| Sanctions et propriété | **GLEIF** | Maisons mères directe et ultime | Quotidienne | Automatique |

Liens, licences et sources écartées : fiche *Projet 6 – Agent finance et marchés : sources des signaux de crise*.

## 1. Configuration

In [ ]:
import warnings
from datetime import date
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import finance_collecte as fc
from finance_collecte import (SIGNAUX, FOURNISSEURS, FOURNISSEURS_COTES, REGLES_UE, MANUELS, FIN_CALIBRATION, ErreurFinance,
                              charger_signal, gazole_ue, federal_register, indicateur, seuil, periode_calibration,
                              contexte_finance, tous_les_signaux, listes_sanctions, correspondances, fiche_fournisseur,
                              PROPRIETE_MANUELLE)

warnings.filterwarnings("ignore", category=DeprecationWarning)   # avertissements internes de numpy/pandas, sans effet sur les résultats
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DEBUT = "2010-01-01"
AUJOURDHUI = pd.Timestamp(date.today())
OUT_DIR = Path("sorties")
OUT_DIR.mkdir(exist_ok=True)
PAYS_GAZOLE = ("EU", "FR", "DE", "IT", "ES", "NL", "BE", "PL")

BLEU, ORANGE, GRIS = "#2a6fbb", "#d9822b", "#8a8f98"
print(f"Période extraite : {DEBUT} → {AUJOURDHUI.date()}  |  seuils calibrés jusqu'au {FIN_CALIBRATION}")

## 2. Les signaux surveillés

Cinq familles, de la plus large à la plus ciblée :

1. **Marchés** (énergie, carbone, fret, change, métaux, prix de production) : ils touchent tous les fournisseurs d'un secteur ou d'un pays. Exemple : le gaz flambe, les fonderies européennes réduisent leur production.
2. **Fournisseurs cotés** : le cours de bourse d'un fournisseur précis. Une chute brutale est souvent le premier signe public qu'il va mal.
3. **Santé des fournisseurs** : une procédure collective (sauvegarde, redressement, liquidation) publiée au BODACC.
4. **Règles et taxes** : un droit de douane, une taxe carbone, une interdiction d'export. Chaque règle a une date de publication et une date d'entrée en vigueur : l'écart entre les deux est la fenêtre pour réagir.
5. **Sanctions et propriété** : le fournisseur, ou sa maison mère, inscrit sur une liste de sanctions (UE ou US).

Les familles 1 et 2 sont des séries de prix : elles ont un seuil d'alerte calibré sur l'historique. Les familles 3 à 5 sont des événements datés : un événement est une alerte en soi.

Mesure des séries : variation sur 20 jours de bourse pour les séries quotidiennes, sur 4 semaines pour le gazole, sur 1 mois pour les séries mensuelles de la Banque mondiale, sur 3 mois pour les prix de production ; le GSCPI est pris en niveau (déjà exprimé en écarts à la moyenne).

In [ ]:
print("Famille 1 – marchés")
display(SIGNAUX[["signal", "nom", "famille", "source", "freq", "sens", "mesure", "unite", "impact"]])
print("Familles 2 à 5 – fournisseurs suivis (bourse si coté, BODACC si SIREN, sanctions et propriété pour tous)")
FOURNISSEURS

## 3. Extraction

Une requête par source ; le cache évite de retélécharger. Un signal qui échoue est signalé et le reste du notebook continue sans lui.

In [ ]:
TOUS = tous_les_signaux()
BRUT = {}
for s in TOUS.itertuples():
    print(f"→ {s.signal:<11}", end=" ")
    try:
        BRUT[s.signal] = charger_signal(s.signal, DEBUT)
        b = BRUT[s.signal]
        print(f"✓  {len(b)} valeurs, {b['date'].min().date()} → {b['date'].max().date()}")
    except ErreurFinance as erreur:
        print(f"{'–' if 'fichier manuel absent' in str(erreur) else '✗'}  {erreur}")

# Gazole par pays (pour comparer les pays européens)
try:
    GAZOLE_PAYS = gazole_ue(PAYS_GAZOLE)
    print(f"\nGazole par pays : {sorted(GAZOLE_PAYS['pays'].unique())}")
except ErreurFinance as erreur:
    GAZOLE_PAYS = pd.DataFrame(columns=["date", "pays", "valeur"])
    print(f"\nGazole par pays ✗ {erreur}")

DISPO = [s for s in TOUS["signal"] if s in BRUT]
print(f"\nSignaux disponibles : {len(DISPO)} / {len(TOUS)}")

### Les règles : Federal Register et suivi UE

Le Federal Register publie chaque texte officiel américain avec sa **date de publication** et sa **date d'entrée en vigueur**. On récupère les règles du Bureau of Industry and Security (Entity List, contrôles d'export) et on cherche le texte qui a visé Wingtech, maison mère de Nexperia.

In [ ]:
def fr_ou_vide(*args, **kwargs):
    try:
        return federal_register(*args, **kwargs)
    except ErreurFinance as erreur:
        print(f"✗ Federal Register : {erreur}")
        return pd.DataFrame(columns=["publication", "entree_en_vigueur", "titre", "type", "url", "document", "agences", "terme", "delai_jours"])

REGLES_US = fr_ou_vide("entity list", "industry-and-security-bureau", debut="2018-01-01")
TARIFS_US = fr_ou_vide("tariff", types=("PRESDOCU",), debut="2025-01-01")
WINGTECH = fr_ou_vide("Wingtech", debut="2024-01-01")
print(f"Règles BIS « entity list » depuis 2018 : {len(REGLES_US)}  |  textes présidentiels sur les droits de douane depuis 2025 : {len(TARIFS_US)}"
      f"  |  textes citant Wingtech : {len(WINGTECH)}")
display(WINGTECH[["publication", "entree_en_vigueur", "titre", "url"]].head())

# La fenêtre pour réagir : jours entre la publication d'une règle et son entrée en vigueur
TOUTES_REGLES = pd.concat([REGLES_US.assign(zone="US", famille="Entity List"), TARIFS_US.assign(zone="US", famille="droits de douane"),
                           REGLES_UE.assign(zone="UE", famille="règle UE", titre=REGLES_UE["regle"],
                                            publication=pd.to_datetime(REGLES_UE["publication"]),
                                            entree_en_vigueur=pd.to_datetime(REGLES_UE["entree_en_vigueur"]))], ignore_index=True)
TOUTES_REGLES["delai_jours"] = (TOUTES_REGLES["entree_en_vigueur"] - TOUTES_REGLES["publication"]).dt.days
display(TOUTES_REGLES.groupby("famille")["delai_jours"].describe()[["count", "min", "50%", "max"]]
        .rename(columns={"count": "règles", "50%": "médiane"}))
display(REGLES_UE)

### Les listes de sanctions

Les sanctions financières de l'UE et la Consolidated Screening List américaine (qui contient l'Entity List), réunies au même format. Chaque fournisseur et chacune de ses maisons mères y sont recherchés plus loin.

In [ ]:
LISTES = listes_sanctions()
print(f"{len(LISTES)} entités sanctionnées, par liste :")
LISTES["liste"].str.split(" : ").str[0].value_counts()

## 4. Qualité et premier aperçu

Avant toute analyse : la période couverte, le rythme réel des données et le plus long trou.

In [ ]:
lignes = []
for sig in DISPO:
    d = BRUT[sig]["date"]
    ecarts = d.diff().dt.days
    lignes.append({"signal": sig, "debut": d.min().date(), "fin": d.max().date(), "valeurs": len(d),
                   "pas_median_jours": ecarts.median(), "plus_long_trou_jours": ecarts.max(),
                   "retard_jours": (AUJOURDHUI - d.max()).days})
QUALITE = pd.DataFrame(lignes).set_index("signal")
QUALITE

In [ ]:
# Tous les signaux en base 100 au 1er janvier 2019 : on voit d'un coup d'œil 2022 et 2026
fig, ax = plt.subplots(figsize=(12, 5))
for sig in DISPO:
    v = BRUT[sig].set_index("date")["valeur"]
    if TOUS.set_index("signal").loc[sig, "mesure"] == "niveau" or sig.startswith("bourse_"):
        continue
    base = v[v.index >= "2019-01-01"]
    if base.empty:
        continue
    ax.plot(base.index, 100 * base / base.iloc[0], lw=1, label=sig)
ax.axhline(100, color=GRIS, lw=0.8)
ax.set_title("Signaux de prix, base 100 au 1er janvier 2019")
ax.legend(ncol=4, fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**À lire** : une hausse de 20 % en un mois est banale pour le gaz européen et exceptionnelle pour l'euro ; une baisse de 30 % est fréquente pour une petite valeur technologique et rare pour un grand équipementier. Les seuils doivent donc être **propres à chaque signal**.

## 5. Indicateurs et seuils propres à chaque signal

**Règle** : un signal est en alerte quand sa variation fait partie des **1 % les plus fortes** observées entre 2010 et 2020 (5 % pour les séries hebdomadaires et mensuelles, qui ont moins de points). Les crises testées ensuite (2022-2025) sont **hors de la période de calibration** : on ne triche pas en fixant le seuil après coup.

In [ ]:
P = TOUS.set_index("signal")
IND, SEUIL = {}, {}
for sig in DISPO:
    p = P.loc[sig]
    IND[sig] = indicateur(BRUT[sig], p.mesure, p.horizon).dropna()
    SEUIL[sig] = seuil(IND[sig], p.sens, p.q)

SEUILS = pd.DataFrame([{"signal": s, "nom": P.loc[s, "nom"], "mesure": "variation %" if P.loc[s, "mesure"] == "variation" else "niveau",
                        "horizon": P.loc[s, "horizon"], "sens": P.loc[s, "sens"], "quantile": P.loc[s, "q"],
                        "seuil": round(SEUIL[s], 2)} for s in DISPO])
SEUILS

In [ ]:
# Calendrier quotidien commun : les séries hebdo et mensuelles gardent leur dernière valeur connue
LIMITE = {"J": 5, "S": 10, "M": 40}
cal = pd.date_range(min(IND[s].index.min() for s in DISPO), AUJOURDHUI, freq="D")
JOURS = pd.DataFrame(index=cal)
for sig in DISPO:
    p = P.loc[sig]
    serie = IND[sig].reindex(cal, method="ffill", limit=LIMITE[p.freq])
    calib = np.sort(periode_calibration(IND[sig]).to_numpy())
    JOURS[sig] = serie
    JOURS[f"alerte_{sig}"] = serie >= SEUIL[sig] if p.sens == "haut" else serie <= SEUIL[sig]
    position = np.searchsorted(calib, serie.to_numpy(), side="right") / len(calib)
    JOURS[f"rang_{sig}"] = np.where(serie.isna(), np.nan, position if p.sens == "haut" else 1 - position)

cols_alerte = [f"alerte_{s}" for s in DISPO]
JOURS["score_finance"] = JOURS[[f"rang_{s}" for s in DISPO]].max(axis=1)
JOURS["alerte_finance"] = JOURS[cols_alerte].any(axis=1)
JOURS.index.name = "date"
print("Score médian sur la calibration (doit être proche de 0,5 pour chaque signal) :")
print(JOURS.loc[:FIN_CALIBRATION, [f"rang_{s}" for s in DISPO]].median().round(2).to_string())

## 6. Validation sur des crises de fournisseurs réelles

Chaque cas est un fournisseur qui a lâché son client. On regarde si un de nos signaux est passé en alerte **dans les 120 jours avant l'impact**, et combien de jours avant.

| Fournisseur | Impact | Ce qui s'est passé | Signal testé |
| --- | --- | --- | --- |
| Nyrstar Budel (zinc, Pays-Bas) | 01/09/2022 | Fonderie mise à l'arrêt à cause du coût de l'énergie, annoncé le 16/08/2022 | Gaz naturel Europe |
| Fournisseurs chinois de Tesla Grünheide | 29/01/2024 | Usine arrêtée du 29/01 au 11/02 faute de composants (détour par le cap de Bonne-Espérance) | GSCPI, fret Drewry |
| Wolfspeed (carbure de silicium) | 30/06/2025 | Faillite (Chapter 11) ; Renesas perd environ 1,7 Md$ sur son acompte | Bourse de Wolfspeed (baisses publiées si Yahoo n'a plus l'historique) |
| Nexperia (puces) | 27/10/2025 | Exportations bloquées, arrêt de production chez Honda | Sanction sur la maison mère, règle US (Federal Register), bourse de Wingtech |

In [ ]:
def episodes(drapeau: pd.Series, ecart_max: int = 7) -> pd.DataFrame:
    """Jours d'alerte regroupés en épisodes (un trou de moins de `ecart_max` jours ne coupe pas l'épisode)."""
    jours = drapeau[drapeau.fillna(False).astype(bool)].index
    if len(jours) == 0:
        return pd.DataFrame(columns=["debut", "fin"])
    coupures = np.where(np.diff(jours.values).astype("timedelta64[D]").astype(int) > ecart_max)[0]
    debuts = [jours[0]] + [jours[i + 1] for i in coupures]
    fins = [jours[i] for i in coupures] + [jours[-1]]
    return pd.DataFrame({"debut": debuts, "fin": fins})


AVANT, APRES = 120, 14   # fenêtre de recherche autour de l'impact (jours)
CAS = pd.DataFrame([
    ("Nyrstar Budel (zinc)",            "2022-09-01", "gaz_europe"),
    ("Fournisseurs de Tesla Grünheide", "2024-01-29", "gscpi"),
    ("Wolfspeed (carbure de silicium)", "2025-06-30", "bourse_wolfspeed"),
    ("Nexperia (puces)",                "2025-10-27", "bourse_wingtech"),
], columns=["cas", "impact", "signal"])
CAS["impact"] = pd.to_datetime(CAS["impact"])

# Quand l'historique boursier a disparu (anciennes actions annulées à la sortie de faillite), on garde les
# baisses publiées dans la presse, avec leur source. Un historique complet placé dans
# data/manuel/bourse_<cle>.csv est utilisé en priorité (finance_collecte.bourse_manuelle).
PUBLIES = {
    "bourse_wolfspeed": ("2025-03-28", -50, "action −50 % après l'échec du refinancement (bondoro.com, résumé du dossier de faillite)"),
}

lignes = []
for c in CAS.itertuples():
    ligne = {"cas": c.cas, "signal": c.signal, "type": "seuil calibré", "impact": c.impact.date(), "seuil": SEUIL.get(c.signal)}
    couvre = c.signal in IND and len(IND[c.signal]) > 0 and IND[c.signal].index.min() <= c.impact - pd.Timedelta(days=AVANT)
    if not couvre and c.signal in PUBLIES:
        d, val, note = PUBLIES[c.signal]
        lignes.append({**ligne, "type": "baisse publiée (historique boursier indisponible)", "seuil": None,
                       "valeur_extreme": val, "alerte_avant_impact": pd.Timestamp(d) <= c.impact,
                       "debut_episode": pd.Timestamp(d).date(), "jours_avance": (c.impact - pd.Timestamp(d)).days, "note": note})
        continue
    if not couvre or c.impact > AUJOURDHUI:
        lignes.append({**ligne, "alerte_avant_impact": False, "note": "non testable (historique trop court)"})
        continue
    ep = episodes(JOURS[f"alerte_{c.signal}"])
    ep = ep[(ep["fin"] >= c.impact - pd.Timedelta(days=AVANT)) & (ep["debut"] <= c.impact + pd.Timedelta(days=APRES))]
    fen = IND[c.signal][(IND[c.signal].index >= c.impact - pd.Timedelta(days=AVANT)) & (IND[c.signal].index <= c.impact)]
    extreme = fen.max() if P.loc[c.signal, "sens"] == "haut" else fen.min()
    premier = ep.iloc[0] if len(ep) else None
    lignes.append({**ligne, "valeur_extreme": round(extreme, 2) if len(fen) else None,
                   "alerte_avant_impact": premier is not None and premier["debut"] <= c.impact,
                   "debut_episode": None if premier is None else premier["debut"].date(),
                   "jours_avance": None if premier is None else (c.impact - premier["debut"]).days, "note": ""})

# Fret Drewry : l'historique complet est payant ; on utilise les valeurs publiées dans les communiqués (USD par conteneur de 40 pieds)
DREWRY = pd.DataFrame([("2023-11-30", 1382), ("2024-01-04", 2670), ("2024-01-11", 3072), ("2024-01-18", 3777), ("2024-01-25", 3964)],
                      columns=["date", "usd_40pieds"])
DREWRY["date"] = pd.to_datetime(DREWRY["date"])
impact_tesla = pd.Timestamp("2024-01-29")
hausse = 100 * (DREWRY.loc[1, "usd_40pieds"] / DREWRY.loc[0, "usd_40pieds"] - 1)
lignes.append({"cas": "Fournisseurs de Tesla Grünheide", "signal": "drewry_wci", "type": "valeurs publiées, sans seuil calibré",
               "impact": impact_tesla.date(), "valeur_extreme": round(hausse, 1), "alerte_avant_impact": True,
               "debut_episode": DREWRY.loc[1, "date"].date(), "jours_avance": (impact_tesla - DREWRY.loc[1, "date"]).days,
               "note": f"fret +{hausse:.0f} % en 5 semaines"})

# Sanction sur la maison mère : date d'inscription de Wingtech dans les listes (UE ou US)
impact_nexperia = pd.Timestamp("2025-10-27")
c = correspondances("Wingtech Technology Co., Ltd.", LISTES)
if len(c) and c["date_inscription"].notna().any():
    d = c["date_inscription"].min()
    lignes.append({"cas": "Nexperia (puces)", "signal": "sanction_maison_mere", "type": "sanction sur la maison mère",
                   "impact": impact_nexperia.date(), "alerte_avant_impact": d <= impact_nexperia, "debut_episode": d.date(),
                   "jours_avance": (impact_nexperia - d).days, "note": f"Wingtech sur {c.loc[c['date_inscription'].idxmin(), 'liste']}"})
else:
    lignes.append({"cas": "Nexperia (puces)", "signal": "sanction_maison_mere", "type": "sanction sur la maison mère",
                   "impact": impact_nexperia.date(), "alerte_avant_impact": False, "note": "Wingtech introuvable dans les listes"})

# Règle : premier texte du Federal Register citant Wingtech, avant l'arrêt Honda
impact_nexperia = pd.Timestamp("2025-10-27")
ligne = {"cas": "Nexperia (puces)", "signal": "federal_register", "type": "règle publiée", "impact": impact_nexperia.date()}
if len(WINGTECH):
    premier = WINGTECH.sort_values("publication").iloc[0]
    ligne.update(alerte_avant_impact=premier["publication"] <= impact_nexperia, debut_episode=premier["publication"].date(),
                 jours_avance=(impact_nexperia - premier["publication"]).days, note=premier["titre"][:80])
else:
    ligne.update(alerte_avant_impact=False, note="Federal Register indisponible : non testable")
lignes.append(ligne)

VALIDATION = pd.DataFrame(lignes).reindex(columns=["cas", "signal", "type", "seuil", "valeur_extreme", "impact",
                                                   "alerte_avant_impact", "debut_episode", "jours_avance", "note"])
VALIDATION["note"] = VALIDATION["note"].fillna("")
VALIDATION

**Lecture** : `jours_avance` est le nombre de jours entre le début de l'épisode d'alerte et l'impact chez le client. Un épisode qui commence plus de 120 jours avant n'est pas compté : avec un seuil à 99 %, des dépassements isolés arrivent par hasard plusieurs fois par an.

In [ ]:
# Chaque cas vu de près : le signal, son seuil (tirets) et l'impact (trait vertical)
cas_graph = [c for c in CAS.itertuples() if c.signal in IND and IND[c.signal].index.min() <= c.impact - pd.Timedelta(days=AVANT)]
if cas_graph:
    fig, axes = plt.subplots(len(cas_graph), 1, figsize=(12, 3.2 * len(cas_graph)), squeeze=False)
    for ax, c in zip(axes[:, 0], cas_graph):
        v = IND[c.signal][(IND[c.signal].index >= c.impact - pd.Timedelta(days=200)) & (IND[c.signal].index <= c.impact + pd.Timedelta(days=60))]
        ax.plot(v.index, v, color=BLEU, lw=1.4, label=P.loc[c.signal, "nom"])
        ax.axhline(SEUIL[c.signal], color=ORANGE, ls="--", lw=1, label="seuil d'alerte")
        ax.axvline(c.impact, color=GRIS, lw=1.2, label="impact chez le client")
        ax.set_title(c.cas)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 7. Le bruit : combien d'alertes par an ?

Un signal qui détecte les crises mais alerte toutes les semaines est inutilisable. On compte les **épisodes d'alerte** par signal et par an, sur toute la période.

In [ ]:
nb_annees = (JOURS.index.max() - JOURS.index.min()).days / 365.25
EPISODES = pd.DataFrame({
    sig: [len(episodes(JOURS[f"alerte_{sig}"])) / nb_annees] for sig in DISPO
}, index=["episodes_par_an"]).T.round(2)
EPISODES["total_jours_en_alerte_par_an"] = [round(JOURS[f"alerte_{s}"].sum() / nb_annees, 1) for s in DISPO]
total = EPISODES["episodes_par_an"].sum()
print(f"Total : {total:.1f} épisodes d'alerte par an, tous signaux confondus")
EPISODES

In [ ]:
par_an = pd.DataFrame({sig: episodes(JOURS[f"alerte_{sig}"])["debut"].dt.year.value_counts() for sig in DISPO}).fillna(0).sort_index()
fig, ax = plt.subplots(figsize=(12, 4.5))
par_an.plot(kind="bar", stacked=True, ax=ax, width=0.8)
ax.set_ylabel("épisodes d'alerte")
ax.set_title("Épisodes d'alerte par année et par signal")
ax.legend(ncol=4, fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

**À lire** : les années de crise (2022, 2025-2026) doivent ressortir nettement ; les autres années donnent le bruit de fond. Si le total dépasse ce qu'un acheteur peut traiter, les signaux de marché doivent servir de **risque de fond** qui fait monter le score d'un fournisseur ; seuls les signaux propres à un fournisseur (sa bourse, une règle qui le vise) déclenchent une alerte critique.

## 8. L'anticipation : combien de semaines entre le Brent et la pompe ?

On compare chaque semaine la variation du Brent et celle du gazole européen, en décalant le gazole de 0 à 12 semaines. Le décalage où la corrélation est la plus forte donne le **délai de transmission** : le temps dont dispose l'acheteur entre le signal et la hausse de ses coûts de transport.

In [ ]:
TRANSMISSION = pd.DataFrame()
if "brent" in BRUT and len(GAZOLE_PAYS):
    brent_hebdo = BRUT["brent"].set_index("date")["valeur"].resample("W-MON").last().pct_change(fill_method=None)
    lignes = []
    for pays, g in GAZOLE_PAYS.groupby("pays"):
        gaz_hebdo = g.set_index("date")["valeur"].resample("W-MON").last().pct_change(fill_method=None)
        for decalage in range(0, 13):
            paire = brent_hebdo.rename("brent").to_frame().join(gaz_hebdo.shift(-decalage).rename("gazole"), how="inner").dropna()
            if len(paire) > 50:
                lignes.append({"pays": pays, "decalage_semaines": decalage, "correlation": paire.corr().iloc[0, 1]})
    CORR = pd.DataFrame(lignes)
    TRANSMISSION = CORR.loc[CORR.groupby("pays")["correlation"].idxmax()].set_index("pays").round(3)
    display(TRANSMISSION)

    fig, ax = plt.subplots(figsize=(10, 4))
    for pays, c in CORR.groupby("pays"):
        ax.plot(c["decalage_semaines"], c["correlation"], marker="o", lw=1, label=pays)
    ax.set_xlabel("décalage du gazole (semaines)")
    ax.set_ylabel("corrélation avec le Brent")
    ax.set_title("Délai de transmission Brent → gazole à la pompe")
    ax.legend(ncol=4, fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Brent ou gazole indisponible : section non calculable.")

## 9. Mode live : la situation aujourd'hui

On applique les mêmes seuils aux dernières valeurs connues, avec le tool `contexte_finance` : c'est exactement ce que recevra l'agent ontologie.

In [ ]:
LIVE = contexte_finance(AUJOURDHUI.date(), pays="EU")
ALERTES_LIVE = pd.DataFrame(LIVE["finance"]["signaux"])
if len(ALERTES_LIVE):
    display(ALERTES_LIVE[["signal", "date_valeur", "valeur_brute", "indicateur", "seuil", "en_alerte", "rang", "impact"]])
print(f"Signaux en alerte aujourd'hui : {LIVE['finance']['nb_alertes']}  |  données manquantes : {LIVE['donnees_manquantes']}")

recentes = TOUTES_REGLES[TOUTES_REGLES["publication"] >= AUJOURDHUI - pd.Timedelta(days=60)]
print(f"Règles publiées ces 60 derniers jours : {len(recentes)}")
display(recentes[["zone", "famille", "publication", "entree_en_vigueur", "delai_jours", "titre"]])

### La fiche de chaque fournisseur

Pour chaque fournisseur : sa bourse, ses maisons mères, les sanctions qui le visent (lui ou une maison mère) et les procédures collectives publiées. C'est ce que l'agent finance transmet au graphe pour chaque nœud fournisseur.

In [ ]:
FICHES = {c: fiche_fournisseur(c, AUJOURDHUI, LISTES) for c in FOURNISSEURS["cle"]}
RESUME_FOURNISSEURS = pd.DataFrame([{
    "fournisseur": f["fournisseur"], "pays": f["pays"],
    "bourse_20j_%": f.get("bourse", {}).get("variation_20j_%"),
    "maisons_meres": ", ".join(p["nom"] for p in f["proprietaires"]) or "-",
    "sanctions": len(f["sanctions"]), "procedures_collectives": len(f.get("procedures_collectives", [])),
    "en_alerte": f["en_alerte"], "alertes": " ; ".join(f["alertes"]),
    "donnees_manquantes": ", ".join(f["donnees_manquantes"])} for f in FICHES.values()])
RESUME_FOURNISSEURS

### Replay fournisseur : Nexperia avant et après l'inscription de sa maison mère

La fiche de Nexperia, demandée la veille puis le lendemain de l'inscription de Wingtech : l'alerte apparaît dès que la sanction est publiée, plus de dix mois avant l'arrêt chez Honda.

In [ ]:
for jour in ("2024-11-30", "2024-12-10"):
    f = fiche_fournisseur("nexperia", jour, LISTES)
    print(f"{jour} : en alerte = {f['en_alerte']}  |  {f['alertes'] or 'aucune alerte'}")

### Replay : ce que l'agent aurait dit pendant une crise

Le même tool, interrogé à une date passée, ne voit que les données disponibles **à cette date** (pas de fuite d'information). Exemple : le 16 août 2022, jour où Nyrstar annonce l'arrêt de sa fonderie de Budel.

In [ ]:
DATE_REPLAY = "2022-08-16"
REPLAY = contexte_finance(DATE_REPLAY, pays="NL")
print(f"Le {DATE_REPLAY}, signaux en alerte : {REPLAY['finance']['nb_alertes']}")
pd.DataFrame(REPLAY["finance"]["signaux"])[["signal", "date_valeur", "indicateur", "seuil", "en_alerte", "impact"]]

## 10. Quelles informations sont utiles ?

Synthèse automatique par signal :

- **cas détectés** : nombre de crises de fournisseurs où le signal a alerté avant l'impact ;
- **meilleure avance** : le plus grand nombre de jours d'avance obtenu ;
- **alertes par an** : le bruit.

In [ ]:
synthese = TOUS.set_index("signal")[["nom", "famille", "impact"]].copy()
synthese["seuil"] = pd.Series(SEUIL).round(2)
synthese["alertes_par_an"] = EPISODES["episodes_par_an"]
v = VALIDATION.copy()
synthese["cas_testes"] = v.groupby("signal").size()
synthese["cas_detectes"] = v[v["alerte_avant_impact"] == True].groupby("signal").size()
synthese["meilleure_avance_jours"] = v.groupby("signal")["jours_avance"].max()
synthese.loc["federal_register", ["nom", "famille", "impact"]] = ["Règles US (Federal Register)", "règles", "Fournisseur coupé ou taxé"]
synthese.loc["drewry_wci", ["nom", "famille", "impact"]] = ["Fret Drewry (valeurs publiées)", "fret", "Délais et coûts d'import d'Asie"]
synthese = synthese.fillna({"cas_testes": 0, "cas_detectes": 0})
synthese.loc["federal_register", "cas_testes"] = (v["signal"] == "federal_register").sum()
synthese.loc["federal_register", "cas_detectes"] = ((v["signal"] == "federal_register") & (v["alerte_avant_impact"] == True)).sum()
synthese.loc["federal_register", "meilleure_avance_jours"] = v.loc[v["signal"] == "federal_register", "jours_avance"].max()
synthese.loc["sanction_maison_mere", ["nom", "famille", "impact"]] = ["Sanction sur la maison mère", "sanctions et propriété", "Fournisseur coupé"]
for sig in ("drewry_wci", "sanction_maison_mere"):
    synthese.loc[sig, "cas_testes"] = (v["signal"] == sig).sum()
    synthese.loc[sig, "cas_detectes"] = ((v["signal"] == sig) & (v["alerte_avant_impact"] == True)).sum()
    synthese.loc[sig, "meilleure_avance_jours"] = v.loc[v["signal"] == sig, "jours_avance"].max()
SYNTHESE = synthese
SYNTHESE

**Comment conclure à partir de ce tableau**

- Un signal qui **détecte ses cas** avec un **bruit modéré** est à garder tel quel.
- Un signal qui détecte mais **alerte trop souvent** est à garder avec un seuil plus strict, ou combiné à un autre (Brent **et** gaz européen, par exemple).
- Un signal **non testé** n'est pas inutile : il lui manque seulement une crise de validation.
- Les **règles** (Federal Register, règles UE) ne se jugent pas au bruit : une règle publiée est un fait, à relier aux fournisseurs concernés.

## 11. Exports pour la suite du projet

In [ ]:
cols_export = DISPO + cols_alerte + ["score_finance", "alerte_finance"]
JOURS[cols_export].reset_index().to_csv(OUT_DIR / "finance_indicateurs_journaliers.csv", index=False)
SEUILS.to_csv(OUT_DIR / "finance_seuils.csv", index=False)
VALIDATION.to_csv(OUT_DIR / "finance_validation_cas.csv", index=False)
EPISODES.to_csv(OUT_DIR / "finance_episodes_par_an.csv")
SYNTHESE.to_csv(OUT_DIR / "finance_synthese_signaux.csv")
TRANSMISSION.to_csv(OUT_DIR / "finance_transmission_brent_gazole.csv")
pd.concat([BRUT[s].assign(signal=s) for s in DISPO]).to_csv(OUT_DIR / "finance_brut.csv", index=False)
GAZOLE_PAYS.to_csv(OUT_DIR / "finance_gazole_pays.csv", index=False)
pd.concat([REGLES_US.assign(zone="US"), WINGTECH.assign(zone="US")]).to_csv(OUT_DIR / "finance_regles_us.csv", index=False)
REGLES_UE.to_csv(OUT_DIR / "finance_regles_ue.csv", index=False)
ALERTES_LIVE.to_csv(OUT_DIR / "finance_alertes_live.csv", index=False)
RESUME_FOURNISSEURS.to_csv(OUT_DIR / "finance_fournisseurs.csv", index=False)
TOUTES_REGLES.to_csv(OUT_DIR / "finance_regles.csv", index=False)
for f in sorted(OUT_DIR.glob("finance_*.csv")):
    print(f"{f.name:<42} {f.stat().st_size / 1e6:6.2f} Mo")

`finance_indicateurs_journaliers.csv` est la table à brancher sur le moteur de scoring : une ligne par jour, avec le `score_finance` (0 à 1) et un drapeau d'alerte par signal. `finance_fournisseurs.csv` donne l'état de chaque fournisseur (bourse, maisons mères, sanctions, procédures collectives). Le gazole par pays (`finance_gazole_pays.csv`) se relie aux sites des fournisseurs par leur code pays.

**Sources** : yfinance (Yahoo, usage personnel), BCE, Commission européenne (Weekly Oil Bulletin), Réserve fédérale de New York (GSCPI), Federal Register (domaine public). Détail des licences dans la fiche des sources.

## 12. Résumé des résultats

Ce résumé est calculé à partir des sections précédentes et se met à jour à chaque exécution.

In [ ]:
lignes = []
for r in VALIDATION.itertuples():
    if r.alerte_avant_impact:
        etat = f"alerte {int(r.jours_avance)} jour(s) avant"
    elif pd.notna(r.jours_avance) and r.jours_avance < 0:
        etat = f"alerte {int(-r.jours_avance)} jour(s) après l'impact (pas d'anticipation)"
    elif r.note:
        etat = r.note
    else:
        etat = "pas d'alerte avant l'impact"
    lignes.append(f"- {r.cas} · {r.type} : {etat}")
print("Validation sur des crises de fournisseurs réelles")
print("\n".join(lignes))
print(f"\nBruit : {EPISODES['episodes_par_an'].sum():.1f} épisodes d'alerte par an pour {len(DISPO)} signaux.")
if len(TRANSMISSION):
    t = TRANSMISSION.loc["EU"] if "EU" in TRANSMISSION.index else TRANSMISSION.iloc[0]
    print(f"Délai Brent → gazole (UE) : {int(t['decalage_semaines'])} semaine(s), corrélation {t['correlation']:.2f}.")
print(f"Signaux de marché en alerte aujourd'hui : {LIVE['finance']['nb_alertes']}.")
print(f"Fournisseurs en alerte aujourd'hui : {int(RESUME_FOURNISSEURS['en_alerte'].sum())} sur {len(RESUME_FOURNISSEURS)}.")

**Version sans code** du notebook :

```bash
jupyter nbconvert --to html --no-input finance_signaux.ipynb
```